# Libraries

In [1]:
# wandb.init(
#     project="messy-mashup",
#     config={
#         "batch_size":32,
#         "learning_rate":0.001,
#         "epochs":10
#     }
# )


# santiy check

In [2]:
import warnings
warnings.filterwarnings("ignore")
import os
import pandas as pd
import random
import numpy as np
import torch
torch.cuda.empty_cache()

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

print("ROOT FILES:")
print(os.listdir(ROOT))


print("\nGENRES:")
GENRE_PATH = os.path.join(ROOT, "genres_stems")
print(os.listdir(GENRE_PATH))


print("\nFIRST GENRE CONTENT:")
first_genre = os.listdir(GENRE_PATH)[0]
print(first_genre)
print(os.listdir(os.path.join(GENRE_PATH, first_genre))[:5])


print("\nFIRST TRACK STRUCTURE:")
track = os.listdir(os.path.join(GENRE_PATH, first_genre))[0]
print(os.listdir(os.path.join(GENRE_PATH, first_genre, track)))


print("\nTEST CSV PREVIEW:")
test_csv = os.path.join(ROOT, "test.csv")
df = pd.read_csv(test_csv)
print(df.head())
print("Total test rows:", len(df))


print("\nESC50 NOISE DATASET:")
noise_path = os.path.join(ROOT, "ESC-50-master", "audio")
print("Noise files:", len(os.listdir(noise_path)))
print(os.listdir(noise_path)[:5])

ROOT FILES:
['ESC-50-master', 'sample_submission.csv', 'genres_stems', 'mashups', 'test.csv']

GENRES:
['disco', 'metal', 'reggae', 'blues', 'rock', 'classical', 'jazz', 'hiphop', 'country', 'pop']

FIRST GENRE CONTENT:
disco
['disco.00052', 'disco.00098', 'disco.00028', 'disco.00069', 'disco.00015']

FIRST TRACK STRUCTURE:
['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

TEST CSV PREVIEW:
   id              filename
0   1  mashups/song0001.wav
1   2  mashups/song0002.wav
2   3  mashups/song0003.wav
3   4  mashups/song0004.wav
4   5  mashups/song0005.wav
Total test rows: 3020

ESC50 NOISE DATASET:
Noise files: 2000
['5-257349-A-15.wav', '5-195557-A-19.wav', '2-122820-B-36.wav', '1-115920-A-22.wav', '1-172649-C-40.wav']


# stage 1


In [3]:
import torch
import torchaudio
import os
import numpy as np
from torch.utils.data import Dataset

ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
GENRE_PATH = os.path.join(ROOT, "genres_stems")
NOISE_PATH = os.path.join(ROOT, "ESC-50-master/audio")

print("Noise files:", len(os.listdir(NOISE_PATH)))


class GenreStemDataset(Dataset):

    def __init__(self, root_dir, noise_dir, sr=22050, duration=30, augment=False):

        self.tracks = []
        self.labels = []
        self.sr = sr
        self.samples = sr * duration
        self.augment = augment

        self.noise_files = [
            os.path.join(noise_dir, f)
            for f in os.listdir(noise_dir)
            if f.endswith(".wav")
        ]

        genres = sorted(os.listdir(root_dir))
        print("Genres detected:", genres)

        for label, genre in enumerate(genres):

            genre_path = os.path.join(root_dir, genre)

            for track in os.listdir(genre_path):

                self.tracks.append(os.path.join(genre_path, track))
                self.labels.append(label)

        print("Total tracks:", len(self.tracks))

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=1024,
            hop_length=512,
            n_mels=128
        )

        self.db_transform = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.tracks)

    def __getitem__(self, idx):

        track_path = self.tracks[idx]
        label = self.labels[idx]

        stems = ["bass.wav", "drums.wav", "other.wav", "vocals.wav"]

        audio_sum = torch.zeros(self.samples)

        for stem in stems:

            file_path = os.path.join(track_path, stem)

            audio, sr = torchaudio.load(file_path)

            audio = torchaudio.functional.resample(audio, sr, self.sr)

            audio = audio.mean(0)

            if len(audio) < self.samples:
                audio = torch.nn.functional.pad(audio, (0, self.samples - len(audio)))
            else:
                audio = audio[:self.samples]

            audio_sum += audio

        # Noise augmentation
        if self.augment and torch.rand(1).item() < 0.5:

            noise_file = np.random.choice(self.noise_files)

            noise, sr = torchaudio.load(noise_file)

            noise = torchaudio.functional.resample(noise, sr, self.sr)

            noise = noise.mean(0)

            if len(noise) < self.samples:
                noise = torch.nn.functional.pad(noise, (0, self.samples - len(noise)))
            else:
                noise = noise[:self.samples]

            noise_level = torch.rand(1).item() * 0.2

            audio_sum = audio_sum + noise_level * noise

        mel = self.mel_transform(audio_sum)

        mel_db = self.db_transform(mel)

        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

        target_len = 1024

        if mel_db.shape[1] < target_len:
            pad = target_len - mel_db.shape[1]
            mel_db = torch.nn.functional.pad(mel_db, (0, pad))
        else:
            start = np.random.randint(0, mel_db.shape[1] - target_len + 1)
            mel_db = mel_db[:, start:start + target_len]

        return mel_db.unsqueeze(0), label

Noise files: 2000


In [4]:
dataset = GenreStemDataset(GENRE_PATH, NOISE_PATH, augment=True)

print("Dataset size:", len(dataset))

x, y = dataset[0]

print("Sample shape:", x.shape)
print("Label:", y)

Genres detected: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Total tracks: 1000
Dataset size: 1000
Sample shape: torch.Size([1, 128, 1024])
Label: 0


# Stage 2

In [5]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from collections import Counter

print("\n----- STAGE 2: DATA SPLIT -----")

indices = list(range(len(dataset)))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=dataset.labels,
    random_state=42
)

train_dataset = Subset(
    GenreStemDataset(GENRE_PATH, NOISE_PATH, augment=True),
    train_idx
)

val_dataset = Subset(
    GenreStemDataset(GENRE_PATH, NOISE_PATH, augment=False),
    val_idx
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))


print("\nClass distribution (train):")
train_labels = [dataset.labels[i] for i in train_idx]
print(Counter(train_labels))


train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2
)


print("\n----- DATALOADER CHECK -----")

x, y = next(iter(train_loader))

print("Train batch shape:", x.shape)
print("Train labels sample:", y[:10])


x, y = next(iter(val_loader))

print("Validation batch shape:", x.shape)
print("Validation labels sample:", y[:10])


----- STAGE 2: DATA SPLIT -----
Genres detected: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Total tracks: 1000
Genres detected: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Total tracks: 1000
Train samples: 800
Validation samples: 200

Class distribution (train):
Counter({5: 80, 2: 80, 9: 80, 4: 80, 7: 80, 8: 80, 6: 80, 0: 80, 1: 80, 3: 80})

----- DATALOADER CHECK -----
Train batch shape: torch.Size([16, 1, 128, 1024])
Train labels sample: tensor([9, 7, 5, 7, 1, 8, 6, 4, 3, 0])
Validation batch shape: torch.Size([8, 1, 128, 1024])
Validation labels sample: tensor([1, 0, 6, 5, 0, 2, 9, 2])


# Stage 3

In [6]:
import torch
import torch.nn as nn
from transformers import ASTForAudioClassification

print("\n----- STAGE 3: INITIALIZING AST MODEL -----")


class ASTModel(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.ast = ASTForAudioClassification.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593"
        )

        hidden = self.ast.config.hidden_size

        self.ast.classifier = nn.Linear(hidden, num_classes)

    def forward(self, x):

        x = x.squeeze(1)      # [B,128,1024]
        x = x.permute(0,2,1)  # [B,1024,128]

        return self.ast(x).logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ASTModel().to(device)
print("Using device:", device)

print("Model initialized successfully")


criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

print("Optimizer + scheduler ready")


print("\n----- FORWARD PASS TEST -----")

x, y = next(iter(train_loader))

print("Input to model:", x.shape)

x = x.to(device)

with torch.no_grad():

    out = model(x)

print("Model output shape:", out.shape)

print("\nStage 3 completed successfully")

2026-03-10 17:34:06.893601: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773164047.180474      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773164047.262481      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773164047.912960      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773164047.913019      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773164047.913022      23 computation_placer.cc:177] computation placer alr


----- STAGE 3: INITIALIZING AST MODEL -----


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Using device: cuda
Model initialized successfully
Optimizer + scheduler ready

----- FORWARD PASS TEST -----
Input to model: torch.Size([16, 1, 128, 1024])
Model output shape: torch.Size([16, 10])

Stage 3 completed successfully


# Stage 4

In [7]:
from sklearn.metrics import f1_score

print("\n----- STAGE 4: TRAINING -----")

def evaluate(model, loader):

    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for audio, labels in loader:

            audio = audio.to(device)
            labels = labels.to(device)

            outputs = model(audio)

            loss = criterion(outputs, labels)

            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average="macro")

    return total_loss / len(loader), f1


epochs = 12
best_loss = float("inf")
patience = 7
counter = 0


for epoch in range(epochs):

    model.train()

    total_loss = 0

    for audio, labels in train_loader:

        audio = audio.to(device)
        labels = labels.to(device)

        outputs = model(audio)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    val_loss, val_f1 = evaluate(model, val_loader)

    scheduler.step()

    print(f"\nEpoch {epoch+1}/{epochs}")
    print("Train Loss:", round(train_loss,4))
    print("Validation Loss:", round(val_loss,4))
    print("Macro F1:", round(val_f1,4))

    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_ast_model.pth")
    else:
        counter += 1

    if counter >= patience:
        print("\nEarly stopping triggered")
        break


----- STAGE 4: TRAINING -----

Epoch 1/12
Train Loss: 1.6023
Validation Loss: 1.1409
Macro F1: 0.7291

Epoch 2/12
Train Loss: 1.0024
Validation Loss: 1.0263
Macro F1: 0.7549

Epoch 3/12
Train Loss: 0.8366
Validation Loss: 0.9027
Macro F1: 0.8537

Epoch 4/12
Train Loss: 0.7204
Validation Loss: 0.8587
Macro F1: 0.8686

Epoch 5/12
Train Loss: 0.6542
Validation Loss: 0.8912
Macro F1: 0.8442

Epoch 6/12
Train Loss: 0.6015
Validation Loss: 0.8584
Macro F1: 0.8632

Epoch 7/12
Train Loss: 0.5642
Validation Loss: 0.8467
Macro F1: 0.8469

Epoch 8/12
Train Loss: 0.5476
Validation Loss: 0.8463
Macro F1: 0.8596

Epoch 9/12
Train Loss: 0.536
Validation Loss: 0.8432
Macro F1: 0.8544

Epoch 10/12
Train Loss: 0.5305
Validation Loss: 0.8352
Macro F1: 0.8606

Epoch 11/12
Train Loss: 0.5279
Validation Loss: 0.8442
Macro F1: 0.8583

Epoch 12/12
Train Loss: 0.5258
Validation Loss: 0.8338
Macro F1: 0.8633


# stage 5


In [8]:
import pandas as pd

print("\n----- STAGE 5: TEST DATASET -----")

class TestDataset(Dataset):

    def __init__(self, root_dir, csv_file, sr=22050, duration=30):

        self.root_dir = root_dir
        self.df = pd.read_csv(csv_file)

        self.sr = sr
        self.samples = sr * duration

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=1024,
            hop_length=512,
            n_mels=128
        )

        self.db_transform = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        track_id = row["id"]

        file_path = os.path.join(self.root_dir, row["filename"])

        audio, sr = torchaudio.load(file_path)

        audio = torchaudio.functional.resample(audio, sr, self.sr)

        audio = audio.mean(0)

        if len(audio) < self.samples:
            audio = torch.nn.functional.pad(audio,(0,self.samples-len(audio)))
        else:
            audio = audio[:self.samples]

        mel = self.mel_transform(audio)

        mel_db = self.db_transform(mel)

        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

        target_len = 1024

        if mel_db.shape[1] < target_len:
            pad = target_len - mel_db.shape[1]
            mel_db = torch.nn.functional.pad(mel_db, (0, pad))
        else:
            start = (mel_db.shape[1] - target_len) // 2
            mel_db = mel_db[:, start:start+target_len]

        return mel_db.unsqueeze(0), track_id


TEST_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
TEST_CSV = os.path.join(TEST_ROOT, "test.csv")

test_dataset = TestDataset(TEST_ROOT, TEST_CSV)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print("Test samples:", len(test_dataset))

x,y = test_dataset[0]

print("Sample shape:", x.shape)
print("Sample id:", y)


----- STAGE 5: TEST DATASET -----
Test samples: 3020
Sample shape: torch.Size([1, 128, 1024])
Sample id: 1


In [9]:
for audio, ids in test_loader:
    print(audio.shape)
    break

torch.Size([8, 1, 128, 1024])


# Stage 6

In [10]:
model.load_state_dict(torch.load("best_ast_model.pth", map_location=device))
model.eval()

ASTModel(
  (ast): ASTForAudioClassification(
    (audio_spectrogram_transformer): ASTModel(
      (embeddings): ASTEmbeddings(
        (patch_embeddings): ASTPatchEmbeddings(
          (projection): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
        )
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (encoder): ASTEncoder(
        (layer): ModuleList(
          (0-11): 12 x ASTLayer(
            (attention): ASTAttention(
              (attention): ASTSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
              )
              (output): ASTSelfOutput(
                (dense): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
            (intermediate): ASTIntermediate(
       

In [11]:
def predict_multi_crop(model, loader):

    model.eval()

    all_preds = []
    all_ids = []

    with torch.no_grad():

        for audio, ids in loader:

            audio = audio.to(device)

            logits_sum = 0
            crops = 12

            for _ in range(crops):

                shift = torch.randint(-120,120,(1,)).item()
                shifted = torch.roll(audio, shifts=shift, dims=3)

                logits = model(shifted)

                logits_sum += logits

            logits_avg = logits_sum / crops

            preds = torch.argmax(logits_avg, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_ids.extend(ids.cpu().numpy())

    return all_preds, all_ids

In [12]:
all_preds, all_ids = predict_multi_crop(model, test_loader)

print("Predictions:", len(all_preds))
print("IDs:", len(all_ids))

Predictions: 3020
IDs: 3020


# stage 7

In [13]:
genre_map = {
0:"blues",
1:"classical",
2:"country",
3:"disco",
4:"hiphop",
5:"jazz",
6:"metal",
7:"pop",
8:"reggae",
9:"rock"
}

submission = pd.DataFrame({
    "id": all_ids,
    "genre": all_preds
})

submission = submission.sort_values("id")

submission["genre"] = submission["genre"].map(genre_map)

submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv created")

print(submission.head())
print(submission.tail())


submission.csv created
   id    genre
0   1      pop
1   2    metal
2   3    disco
3   4    metal
4   5  country
        id      genre
3015  3016       rock
3016  3017  classical
3017  3018     reggae
3018  3019    country
3019  3020  classical


In [14]:
print(submission.shape)
print(submission.head())
print(submission.tail())

(3020, 2)
   id    genre
0   1      pop
1   2    metal
2   3    disco
3   4    metal
4   5  country
        id      genre
3015  3016       rock
3016  3017  classical
3017  3018     reggae
3018  3019    country
3019  3020  classical


# Stage 1

In [15]:
# !pip install transformers -q

# import warnings
# warnings.filterwarnings("ignore")

# import os
# import torch
# import pandas as pd
# import torchaudio
# import numpy as np
# import torch.nn as nn
# import torch.optim as optim

# from transformers import ASTForAudioClassification
# from torch.utils.data import Dataset, DataLoader
# from sklearn.metrics import f1_score

# print("Libraries loaded")

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Using device:", device)


# class GenreStemDataset(Dataset):

#     def __init__(self, root_dir, sr=22050, duration=30, augment=False):

#         self.tracks = []
#         self.labels = []
#         self.sr = sr
#         self.samples = sr * duration
#         self.augment = augment

#         self.mel_transform = torchaudio.transforms.MelSpectrogram(
#             sample_rate=sr,
#             n_fft=1024,
#             hop_length=512,
#             n_mels=128
#         )

#         self.db_transform = torchaudio.transforms.AmplitudeToDB()

#         # SORT folders for stable label mapping
#         genres = sorted(os.listdir(root_dir))

#         print("Genres detected:", genres)

#         for label, genre in enumerate(genres):

#             genre_path = os.path.join(root_dir, genre)

#             for track in os.listdir(genre_path):

#                 self.tracks.append(os.path.join(genre_path, track))
#                 self.labels.append(label)

#     def __len__(self):
#         return len(self.tracks)

#     def __getitem__(self, idx):

#         track_path = self.tracks[idx]
#         label = self.labels[idx]

#         stems = ["bass.wav","drums.wav","other.wav","vocals.wav"]

#         audio_sum = torch.zeros(self.samples)

#         for stem in stems:

#             file_path = os.path.join(track_path, stem)

#             audio, sr = torchaudio.load(file_path)

#             audio = torchaudio.functional.resample(audio, sr, self.sr)

#             audio = audio.mean(0)

#             if len(audio) < self.samples:
#                 audio = torch.nn.functional.pad(audio,(0,self.samples-len(audio)))
#             else:
#                 audio = audio[:self.samples]

#             audio_sum += audio

#         # MEL
#         mel = self.mel_transform(audio_sum)

#         mel_db = self.db_transform(mel)
#         mel_db = (mel_db - mel_db.mean())/(mel_db.std()+1e-6)

#         # FORCE SIZE FOR AST
#         mel_db = torch.nn.functional.interpolate(
#             mel_db.unsqueeze(0).unsqueeze(0),
#             size=(128,1024),
#             mode="bilinear",
#             align_corners=False
#         ).squeeze()

#         # SPEC AUGMENT
#         if self.augment:

#             t = mel_db.shape[-1]

#             t_mask = torch.randint(10,30,(1,)).item()
#             t0 = torch.randint(0,max(1,t-t_mask),(1,)).item()

#             mel_db[:,t0:t0+t_mask] = mel_db.min()

#             f = mel_db.shape[-2]

#             f_mask = torch.randint(5,20,(1,)).item()
#             f0 = torch.randint(0,max(1,f-f_mask),(1,)).item()

#             mel_db[f0:f0+f_mask,:] = mel_db.min()

#         return mel_db.unsqueeze(0), label


# Path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

# dataset = GenreStemDataset(Path, augment=False)

# print("Dataset size:", len(dataset))

# audio,label = dataset[0]

# print("Sample audio shape:", audio.shape)
# print("Sample label:", label)

In [16]:
# for i in range(5):
#     x,y = dataset[i]
#     print(i, x.shape, y)

In [17]:
# from collections import Counter
# print(Counter(dataset.labels))

In [18]:
# for i in [0,120,250,350,450,550,650,750,850,950]:
#     x,y = dataset[i]
#     print(i, y)

In [19]:
# Path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
# dataset = GenreStemDataset(Path,augment=False)

# print(len(dataset))

# audio, label = dataset[656]
# print(audio.shape, label)

# Stage 2


In [20]:
# # ==============================
# # STAGE 2: TRAIN/TEST SPLIT
# # ==============================

# from torch.utils.data import DataLoader, Subset
# from sklearn.model_selection import train_test_split

# print("\n----- STAGE 2: DATA SPLIT -----")

# indices = list(range(len(dataset)))

# train_idx, test_idx = train_test_split(
#     indices,
#     test_size=0.2,
#     stratify=dataset.labels,
#     random_state=42
# )

# print("Train samples:", len(train_idx))
# print("Validation samples:", len(test_idx))


# # ==============================
# # CREATE SUBSETS
# # ==============================

# train_full = GenreStemDataset(Path, augment=True)
# test_full  = GenreStemDataset(Path, augment=False)

# train_dataset = Subset(train_full, train_idx)
# test_dataset  = Subset(test_full, test_idx)

# print("\nAugmentation status:")
# print("Train augment:", train_dataset.dataset.augment)
# print("Test augment:", test_dataset.dataset.augment)


# # ==============================
# # DATALOADERS
# # ==============================

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=16,
#     shuffle=True,
#     num_workers=2,
#     pin_memory=True
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=8,
#     shuffle=False,
#     num_workers=2
# )

# print("\n----- DATALOADER CHECK -----")

# # verify train loader
# for audio, label in train_loader:
#     print("Train batch shape:", audio.shape)
#     print("Train labels sample:", label[:10])
#     break

# # verify validation loader
# for audio, label in test_loader:
#     print("Validation batch shape:", audio.shape)
#     print("Validation labels sample:", label[:10])
#     break


# # ==============================
# # STAGE 2: AST MODEL
# # ==============================

# print("\n----- INITIALIZING AST MODEL -----")

# class ASTModel(nn.Module):

#     def __init__(self, num_classes=10):
#         super().__init__()

#         self.ast = ASTForAudioClassification.from_pretrained(
#             "MIT/ast-finetuned-audioset-10-10-0.4593"
#         )

#         # replace classifier
#         self.ast.classifier = nn.Linear(
#             self.ast.config.hidden_size,
#             num_classes
#         )

#     def forward(self, x):

#         # input: [B,1,128,1024]
#         # print("Input to model:", x.shape)

#         x = x.squeeze(1)          # [B,128,1024]
#         x = x.permute(0,2,1)      # [B,1024,128]

#         # print("Input to AST:", x.shape)

#         return self.ast(x).logits


# model = ASTModel().to(device)

# print("\nModel initialized successfully")


# # ==============================
# # LOSS + OPTIMIZER
# # ==============================

# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=1e-5
# )

# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer,
#     mode="min",
#     factor=0.5,
#     patience=3
# )

# print("\nOptimizer + scheduler ready")


# # ==============================
# # QUICK FORWARD PASS TEST
# # ==============================

# print("\n----- FORWARD PASS TEST -----")

# audio, label = next(iter(train_loader))

# audio = audio.to(device)

# with torch.no_grad():
#     output = model(audio)

# print("Model output shape:", output.shape)

# print("\nStage 2 completed successfully!")

# Stage 3

In [21]:
# print("\n----- STAGE 3: TRAINING -----")

# epochs = 35
# best_loss = float("inf")

# for epoch in range(epochs):

#     model.train()
#     total_loss = 0

#     for audio,label in train_loader:

#         audio = audio.to(device)
#         label = label.to(device)

#         optimizer.zero_grad()

#         output = model(audio)

#         loss = criterion(output,label)

#         loss.backward()

#         # important for transformers
#         torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

#         optimizer.step()

#         total_loss += loss.item()

#     train_loss = total_loss / len(train_loader)

#     # ------------------------
#     # VALIDATION
#     # ------------------------

#     model.eval()
#     val_loss = 0
#     all_preds=[]
#     all_labels=[]

#     with torch.no_grad():

#         for audio,label in test_loader:

#             audio = audio.to(device)
#             label = label.to(device)

#             output = model(audio)

#             loss = criterion(output,label)

#             val_loss += loss.item()

#             preds = torch.argmax(output,dim=1)

#             all_preds.extend(preds.cpu().numpy())
#             all_labels.extend(label.cpu().numpy())

#     val_loss = val_loss / len(test_loader)

#     f1 = f1_score(all_labels,all_preds,average="macro")

#     scheduler.step(val_loss)

#     print(f"\nEpoch {epoch+1}/{epochs}")
#     print("Train Loss:",round(train_loss,4))
#     print("Validation Loss:",round(val_loss,4))
#     print("Macro F1:",round(f1,4))
# print("Stage 3 completed successfully!")

In [22]:
# for audio, labels in train_loader:
#     print(audio.shape)
#     print(labels)
#     break

In [23]:
# x,y = dataset[0]
# print(x.shape)

## Simple CNN Model

In [24]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class GenreCNN(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.features = nn.Sequential(
            
#             nn.Conv2d(1,16,kernel_size=2,padding=1),
#             nn.BatchNorm2d(16),
#             nn.LeakyReLU(0.1),
            
#             nn.Conv2d(16,32,kernel_size=2,padding=1),
#             nn.BatchNorm2d(32),
#             nn.LeakyReLU(0.1),

#             nn.MaxPool2d(2),
#             nn.Dropout(0.2),


            
#             nn.Conv2d(32,64,kernel_size=3,padding=1),
#             nn.BatchNorm2d(64),
#             nn.LeakyReLU(0.1),
            
#             nn.Conv2d(64,128,kernel_size=3,padding=1),
#             nn.BatchNorm2d(128),
#             nn.LeakyReLU(0.1),

#             nn.MaxPool2d(2),
#             nn.Dropout(0.2),


            
#             nn.Conv2d(128,128,kernel_size=3,padding=1),
#             nn.BatchNorm2d(128),
#             nn.LeakyReLU(0.1),

#             nn.MaxPool2d(2),
#         )
#         self.pool = nn.AdaptiveAvgPool2d((1,1))
#         self.classifier = nn.Sequential(
#             nn.Linear(128,128),
#             nn.LeakyReLU(0.1),
#             nn.Dropout(0.4),
#             nn.Linear(128,10)
#         )

#     def forward(self, x):
#         x = self.features(x)
#         x = self.pool(x)   
#         x = torch.flatten(x,1)
#         x = self.classifier(x)
        
#         return x


In [25]:
# class ASTModel(nn.Module):

#     def __init__(self, num_classes=10):

#         super().__init__()

#         self.ast = ASTForAudioClassification.from_pretrained(
#             "MIT/ast-finetuned-audioset-10-10-0.4593"
#         )

#         self.ast.classifier = nn.Linear(self.ast.config.hidden_size, num_classes)

#     def forward(self, x):

#         x = x.squeeze(1)        # [B,128,1024]
#         x = x.permute(0,2,1)    # [B,1024,128]

#         return self.ast(x).logits

## Training

In [26]:
# model = GenreCNN().to(device)
model = ASTModel().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

In [27]:
# def evalucate(model,loader):
#     model.eval()
#     total_loss = 0
#     all_preds=[]
#     all_labels=[]
#     with torch.no_grad():
#         for audio, label in loader:
            
#             audio = audio.to(device)
#             label = label.to(device)
            
#             output = model(audio)
#             loss = criterion(output, label)
            
#             preds = torch.argmax(output, dim=1) 
#             all_preds.extend(preds.cpu().numpy())
#             all_labels.extend(label.cpu().numpy())
#             total_loss += loss.item()

#         f1 = f1_score(all_labels, all_preds, average='macro')
#         total = total_loss / len(loader)
#         return total, f1
            
    

# best_loss = float('inf')
# epochs = 20
# patience = 7 
# counter = 0

# for epoch in range(epochs):
    
#     model.train()
#     total_loss = 0

#     for audio, label in train_loader:

#         audio = audio.to(device)
#         label = label.to(device)

#         output = model(audio)

#         loss = criterion(output, label)

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()
#     train_loss = total_loss/len(train_loader)
    
#     scheduler.step(train_loss)
#     current_lr = optimizer.param_groups[0]['lr']
#     val_total, f1 = evalucate(model, test_loader)

    
#     print(f"Epoch: {epoch+1}")
#     print(f"Train Loss: {train_loss:.4f}")
#     print(f"Final LR: {current_lr:.4f}")
#     print(f"F1 Score: {f1:.4f}")
#     print(f"Validation Loss: {val_total:.4f}")

    
#     if val_total < best_loss:
#         best_loss = val_total
#         counter = 0
#     else:
#         counter += 1

#     if counter >= patience:
#         print("Early Stopping")
#         break

In [28]:
# torch.save(model.state_dict(),"genre_model.pth")
# wandb.save("genre_model.pth")

## Predict on train mashups

# Stage 4

In [29]:
# class TestStemDataset(Dataset):

#     def __init__(self, root_dir, csv_file, sr=22050, duration=30):
#         self.root_dir = root_dir
#         self.df = pd.read_csv(csv_file)
#         self.sr = sr
#         self.samples = sr * duration


#         self.mel_transform = torchaudio.transforms.MelSpectrogram(
#             sample_rate=sr,
#             n_fft=1024,
#             hop_length=512,
#             n_mels=128
#         )

#         self.db_transform = torchaudio.transforms.AmplitudeToDB()

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):

#         row = self.df.iloc[idx]

#         track_id = row["id"]
#         file_path = os.path.join(self.root_dir, row["filename"])

#         audio, sr = torchaudio.load(file_path)

#         audio = torchaudio.functional.resample(audio, sr, self.sr)
#         audio = audio.mean(0)

#         if len(audio) < self.samples:
#             audio = torch.nn.functional.pad(audio,(0,self.samples-len(audio)))
#         else:
#             audio = audio[:self.samples]

#         mel = self.mel_transform(audio) # audio will or something else

#         mel_db = self.db_transform(mel)
#         mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

#         target_len = 1024
#         if mel_db.shape[1] < target_len:
#             pad = target_len - mel_db.shape[1]
#             mel_db = torch.nn.functional.pad(mel_db, (0, pad))
#         else:
#             start = (mel_db.shape[1] - target_len) // 2
#             mel_db = mel_db[:, start:start+target_len]

#         return mel_db.unsqueeze(0), track_id


In [30]:
# TEST_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
# TEST_CSV = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
# print(pd.read_csv(TEST_CSV).head())
# test_dataset = TestStemDataset(TEST_ROOT, TEST_CSV)

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=4,
#     shuffle=False
# )
# print("done")

In [31]:
# def predict_multi_crop(model, dataset, num_crops=6):

#     model.eval()

#     all_preds = []
#     all_ids = []

#     with torch.no_grad():

#         for idx in range(len(dataset)):

#             audio, track_id = dataset[idx]

#             audio = audio.to(device)

#             logits_sum = 0

#             for _ in range(num_crops):

#                 start = torch.randint(0, 512, (1,)).item()

#                 crop = audio[:, :, start:start+512]

#                 crop = torch.nn.functional.interpolate(
#                     crop.unsqueeze(0),
#                     size=(128,1024),
#                     mode="bilinear",
#                     align_corners=False
#                 ).squeeze(0)

#                 crop = crop.unsqueeze(0).to(device)

#                 output = model(crop)

#                 logits_sum += output

#             logits_avg = logits_sum / num_crops

#             pred = torch.argmax(logits_avg, dim=1).item()

#             all_preds.append(pred)
#             all_ids.append(track_id)

#     return all_preds, all_ids
    
# print("Running multi-crop inference...")

# all_preds, all_labels = predict_multi_crop(model, test_dataset, num_crops=6)

# Submission

In [32]:
# print("\n----- TEST DATA CHECK -----")

# x, y = test_dataset[0]
# print("Test sample shape:", x.shape)
# print("Sample ID:", y)

# print("\nTotal test samples:", len(test_dataset))


# # print("\n----- RUNNING INFERENCE -----")

# # all_preds, all_labels = final_test(model, test_loader)

# print("Predictions:", len(all_preds))
# print("IDs:", len(all_labels))


# # ----------------------------
# # GENRE MAP
# # ----------------------------

# genre_map = {
# 0:"blues",
# 1:"classical",
# 2:"country",
# 3:"disco",
# 4:"hiphop",
# 5:"jazz",
# 6:"metal",
# 7:"pop",
# 8:"reggae",
# 9:"rock"
# }


# # ----------------------------
# # CREATE SUBMISSION
# # ----------------------------

# submission = pd.DataFrame({
#     "id": all_labels,
#     "genre": all_preds
# })

# submission = submission.sort_values("id")

# submission["genre"] = submission["genre"].map(genre_map)


# print("\n----- SUBMISSION CHECK -----")

# print("Submission rows:", len(submission))
# print(submission.head())
# print(submission.tail())


# submission.to_csv("submission.csv", index=False)

# print("\nsubmission.csv created successfully")

In [33]:
# # convert numeric labels → genre names
# genre_map = {
#     0:"blues",
#     1:"classical",
#     2:"country",
#     3:"disco",
#     4:"hiphop",
#     5:"jazz",
#     6:"metal",
#     7:"pop",
#     8:"reggae",
#     9:"rock"
# }

In [34]:
# submission = pd.DataFrame({
#     "id": all_labels,
#     "genre": all_preds
# })

# submission = submission.sort_values("id")

# submission["genre"] = submission["genre"].map(genre_map)

# submission.to_csv("submission.csv", index=False)

# print("submission.csv created successfully")
# submission.head(10)

In [35]:
# print(len(all_preds))
# print(len(all_labels))
# print(submission.head())

In [36]:
# print(sorted(os.listdir(Path)))

In [37]:
# print(submission.shape)
# print(submission.head())
# print(submission.tail())

In [38]:
# pd.read_csv("submission.csv").head()

In [39]:
# !head submission.csv

# **Milestone 1**

In [40]:
# import pandas as pd

In [41]:
# pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv").head()


In [42]:
# pd.read_csv("submission.csv").head()

In [43]:
# import os
# import glob
# import numpy as np
# import pandas as pd
# from tqdm import tqdm
# import librosa
# import librosa.display
# import matplotlib.pyplot as plt
# import random
# import torch

# import warnings
# warnings.filterwarnings("ignore")

In [44]:
# #----------------------------- DON'T CHANGE THIS --------------------------
# DATA_SEED = 67
# TRAINING_SEED = 1234
# SR = 22050
# DURATION = 5.0
# N_FFT = 2048
# HOP_LENGTH = 512
# N_MELS = 128
# TOP_DB=20
# TARGET_SNR_DB = 10

# random.seed(DATA_SEED)
# np.random.seed(DATA_SEED)
# torch.manual_seed(DATA_SEED)
# torch.cuda.manual_seed(DATA_SEED)

In [45]:
# # CONFIGURATION
# DATA_ROOT ='/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
# GENRES = sorted(os.listdir(DATA_ROOT))
# STEMS = {"drums.wav": "drums", "vocals.wav": "vocals", "bass.wav": "bass", "other.wav": "other"}
# STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
# GENRE_TO_TEST = 'rock'
# SONG_INDEX = 0 # Enter index as per Q10.

In [46]:
# print(os.listdir(DATA_ROOT)[:3])
# sample_genre = GENRES[0]
# sample_genre_path = os.path.join(DATA_ROOT, sample_genre)

# print("Example songs:", os.listdir(sample_genre_path)[:3])
# sample_song = os.listdir(sample_genre_path)[0]
# sample_song_path = os.path.join(sample_genre_path, sample_song)

# print("Stem files:", os.listdir(sample_song_path))
# import os

# stem_path = os.path.join(sample_song_path, "drums.wav")
# print("File size:", os.path.getsize(stem_path))


In [47]:
# def build_dataset(root_dir, val_split=0.17, seed=42):
    
#     train_dataset = {g: {v: [] for v in STEMS.values()} for g in GENRES}
#     val_dataset   = {g: {v: [] for v in STEMS.values()} for g in GENRES}

#     rng = random.Random(seed)

#     for genre in GENRES:
#         genre_path = os.path.join(root_dir, genre)

#         if not os.path.isdir(genre_path):
#             continue

#         all_songs = os.listdir(genre_path)
#         valid_songs = []

#         for song in all_songs:
#             song_path = os.path.join(genre_path, song)

#             if not os.path.isdir(song_path):
#                 continue

#             stem_files = os.listdir(song_path)

#             # Check completeness
#             if not set(STEMS.keys()).issubset(set(stem_files)):
#                 continue

#             # Check corruption (size > 4KB)
#             corrupted = False
#             for stem_file in STEMS.keys():
#                 file_path = os.path.join(song_path, stem_file)
#                 if os.path.getsize(file_path) < 4096:
#                     corrupted = True
#                     break

#             if corrupted:
#                 continue

#             valid_songs.append(song)

#         # Shuffle
#         rng.shuffle(valid_songs)

#         split_idx = int(len(valid_songs) * (1 - val_split))
#         train_songs = valid_songs[:split_idx]
#         val_songs   = valid_songs[split_idx:]

#         # Populate dictionary
#         for song in train_songs:
#             song_path = os.path.join(genre_path, song)
#             for stem_file, stem_name in STEMS.items():
#                 file_path = os.path.join(song_path, stem_file)
#                 train_dataset[genre][stem_name].append(file_path)

#         for song in val_songs:
#             song_path = os.path.join(genre_path, song)
#             for stem_file, stem_name in STEMS.items():
#                 file_path = os.path.join(song_path, stem_file)
#                 val_dataset[genre][stem_name].append(file_path)

#     return train_dataset, val_dataset


In [48]:
# tr, val = build_dataset(DATA_ROOT)

# for g in GENRES:
#     print(g,
#           "Train:", len(tr[g]['drums']),
#           "Val:", len(val[g]['drums']))


In [49]:
# SR = 22050
# DURATION = 5.0
# TOP_DB = 20

In [50]:
# import librosa
# import numpy as np
# import pandas as pd

# def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
#     """
#     Input:
#         dataset_dict: {genre: {stem: [paths...]}}
#     Output:
#         DataFrame of files with silence >= threshold_sec
#     """

#     records = []
#     total_files = 0

#     for genre, stems_dict in dataset_dict.items():
#         for stem_name, file_list in stems_dict.items():
#             for file_path in file_list:
#                 total_files += 1

#                 # Load FULL audio
#                 y, _ = librosa.load(file_path, sr=sr)

#                 total_duration = len(y) / sr

#                 # Find non-silent intervals
#                 intervals = librosa.effects.split(y, top_db=top_db)

#                 silence_type = []
#                 max_silence = 0.0

#                 # CASE A: Fully silent
#                 if len(intervals) == 0:
#                     max_silence = total_duration
#                     silence_type.append("FULL")

#                 else:
#                     # Calculate silence segments
#                     silence_segments = []

#                     # Start silence
#                     if intervals[0][0] > 0:
#                         silence_segments.append(intervals[0][0] / sr)
#                         silence_type.append("START")

#                     # Middle silence
#                     for i in range(1, len(intervals)):
#                         gap = (intervals[i][0] - intervals[i-1][1]) / sr
#                         if gap > 0:
#                             silence_segments.append(gap)
#                             silence_type.append("MIDDLE")

#                     # End silence
#                     if intervals[-1][1] < len(y):
#                         silence_segments.append((len(y) - intervals[-1][1]) / sr)
#                         silence_type.append("END")

#                     if silence_segments:
#                         max_silence = max(silence_segments)

#                 # Store only if silence ≥ threshold
#                 if max_silence >= threshold_sec:
#                     records.append({
#                         "Genre": genre,
#                         "Stem": stem_name,
#                         "Duration": round(total_duration, 2),
#                         "Max_Silence_Sec": round(max_silence, 2),
#                         "Silence_Location": ", ".join(sorted(set(silence_type))),
#                         "File_Path": file_path
#                     })

#     print(f"Total files checked: {total_files}")

#     df = pd.DataFrame(records)
#     return df

In [51]:
# df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

# pivot = pd.pivot_table(df_silence, index="Genre", columns="Stem",values="File_Path", aggfunc="count", fill_value=0)

# print(pivot)

In [52]:
# stems_audio = []
# try:
#     for key in STEM_KEYS:
#         # Get file path for selected song
#         file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]

#         # Load only 5 seconds
#         y, _ = librosa.load(file_path, sr=SR, duration=DURATION)

#         stems_audio.append(y)
#     print("Audio loaded successfully.")
# except NameError:
#     print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
# except IndexError:
#     print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
# except Exception as e:
#     print(f"ERROR: {e}")

In [53]:
# # Stack them into numpy array (Shape: 4 x Samples)
# stems_stack = np.vstack(stems_audio)

# # Mix stems by summing element-wise
# mix_raw = np.sum(stems_stack, axis=0)

# # Calculate RMS manually
# rms_val = np.sqrt(np.mean(mix_raw ** 2))

# # Peak normalization
# max_val = np.max(np.abs(mix_raw))

# if max_val > 0:
#     mix_norm = mix_raw / max_val
# else:
#     mix_norm = mix_raw

# # VALIDATION
# assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."


In [54]:
# print("Stack shape:", stems_stack.shape)
# print("Mix length:", len(mix_raw))
# print("RMS:", rms_val)
# print("Max after norm:", np.max(np.abs(mix_norm)))


1. Complete the function `build_dataset` in question notebook and answer following questions (Q1 to Q3).
Hint: 1kb = 1024 bytes

What is the value of total number of corrupted songs ( less than 4kb) + (total number of songs < 5.0491MB)

In [55]:
# import os

# corrupted_count = 0
# small_5mb_count = 0

# threshold_4kb = 4 * 1024
# threshold_5mb_low = 5.0491 * 1024 * 1024  # convert MB to bytes

# for genre in GENRES:
#     genre_path = os.path.join(DATA_ROOT, genre)

#     for song in os.listdir(genre_path):
#         song_path = os.path.join(genre_path, song)

#         if not os.path.isdir(song_path):
#             continue

#         for stem in STEMS.keys():
#             file_path = os.path.join(song_path, stem)
#             size = os.path.getsize(file_path)

#             if size < threshold_4kb:
#                 corrupted_count += 1

#             if size < threshold_5mb_low:
#                 small_5mb_count += 1

# print("Corrupted (<4KB):", corrupted_count)
# print("Less than 5.0491MB:", small_5mb_count)
# print("Sum:", corrupted_count + small_5mb_count)


2. What is the absolute difference between  total number of songs > 5.0493MB and total number of songs < 5.0491MB ?

In [56]:
# # total songs > 5.0493MB
# # AND
# # total songs < 5.0491MB

# threshold_5mb_high = 5.0493 * 1024 * 1024

# greater_count = 0
# less_count = 0

# for genre in GENRES:
#     genre_path = os.path.join(DATA_ROOT, genre)

#     for song in os.listdir(genre_path):
#         song_path = os.path.join(genre_path, song)

#         if not os.path.isdir(song_path):
#             continue

#         for stem in STEMS.keys():
#             file_path = os.path.join(song_path, stem)
#             size = os.path.getsize(file_path)

#             if size > threshold_5mb_high:
#                 greater_count += 1

#             if size < threshold_5mb_low:
#                 less_count += 1

# print("Absolute Difference:", abs(greater_count - less_count))


3. What is the absolute difference between the number of training reggae drum samples and the number of validation country vocal samples?

In [57]:
# train_reggae_drums = len(tr['reggae']['drums'])
# val_country_vocals = len(val['country']['vocals'])

# print(abs(train_reggae_drums - val_country_vocals))


4. Complete the function `find_long_silences` in question notebook and answer following questions (Q4 to Q9).

Total number of sound files having silence greater than equal to 5 secs.

In [58]:
# df_silence = find_long_silences(tr, threshold_sec=5.0)

In [59]:
# print(len(df_silence))

5. Total number of sound tracks in Vocals where silence >= 5 secs

In [60]:
# print(len(df_silence[df_silence['Stem'] == 'vocals']))

6. What's the average Silence Length in Vocals (in secs).

In [61]:
# vocals_df = df_silence[df_silence['Stem'] == 'vocals']
# print(round(vocals_df['Max_Silence_Sec'].mean(), 2))

7. Total number of drums sound tracks in jazz where silence >= 5 secs

In [62]:
# jazz_drums = df_silence[
#     (df_silence['Genre'] == 'jazz') &
#     (df_silence['Stem'] == 'drums')
# ]

# print(len(jazz_drums))

8. Total number of drums sound tracks in jazz where silence >= 5 secs and Silence_Location is only middle.

In [63]:
# jazz_middle = jazz_drums[
#     jazz_drums['Silence_Location'] == 'MIDDLE'
# ]

# print(len(jazz_middle))


In [64]:
# jazz_middle_only = df_silence[
#     (df_silence['Genre'] == 'jazz') &
#     (df_silence['Stem'] == 'drums') &
#     (df_silence['Silence_Location'] == 'MIDDLE')
# ]

# print(len(jazz_middle_only))


In [65]:
# print(df_silence[
#     (df_silence['Genre'] == 'jazz') &
#     (df_silence['Stem'] == 'drums')
# ]['Silence_Location'].unique())


In [66]:
# jazz_middle_only = df_silence[
#     (df_silence['Genre'] == 'jazz') &
#     (df_silence['Stem'] == 'drums') &
#     (~df_silence['Silence_Location'].str.contains("START")) &
#     (~df_silence['Silence_Location'].str.contains("END")) &
#     (df_silence['Silence_Location'].str.contains("MIDDLE"))
# ]

# print(len(jazz_middle_only))


9. Total number of drums sound tracks in jazz where silence >= 5 secs and Max_Silence_Sec >= 10

In [67]:
# jazz_10 = jazz_drums[
#     jazz_drums['Max_Silence_Sec'] >= 10
# ]

# print(len(jazz_10))


10. Select the first song from the ‘rock’ genre, combine all stems to prepare a sample, perform the tasks outlined in the notebook, and answer questions 10–12 based on the results.

What is the length of the mix sample?

In [68]:
# print(len(mix_raw))

11. What is the value of RMS Amplitude of mix sample?

In [69]:
# print(round(rms_val, 2))

12. What is the value of max value of peak  normalized sample ?

In [70]:
# print("Raw Max:", round(np.max(np.abs(mix_raw)), 2))
# print("Norm Max:", round(np.max(np.abs(mix_norm)), 2))


# Milestone 2

In [71]:
# durations = []

# for stem_name, file_list in tr['jazz'].items():
#     for file_path in file_list:
#         y, _ = librosa.load(file_path, sr=SR)
#         duration = len(y) / SR
#         durations.append(duration)

# mean_duration = np.mean(durations)

# print("Mean duration (seconds):", round(mean_duration, 2))

In [72]:
# import librosa
# import os

# sample_rates = set()

# for genre in GENRES:
#     genre_path = os.path.join(DATA_ROOT, genre)

#     for song in os.listdir(genre_path):
#         song_path = os.path.join(genre_path, song)

#         if not os.path.isdir(song_path):
#             continue

#         for stem_file in STEMS.keys():
#             file_path = os.path.join(song_path, stem_file)

#             # Load WITHOUT resampling
#             _, sr = librosa.load(file_path, sr=None)
#             sample_rates.add(sr)

# print("Unique Sample Rates:", sorted(sample_rates))

In [73]:
# import os

# corrupted_count = 0

# for genre in tr:
#     for stem_name in tr[genre]:
#         for file_path in tr[genre][stem_name]:
#             if os.path.getsize(file_path) == 0:
#                 corrupted_count += 1

# print("Zero-byte files in train:", corrupted_count)


In [74]:
# corrupted_count = 0

# for genre in tr:
#     for stem_name in tr[genre]:
#         for file_path in tr[genre][stem_name]:
#             if os.path.getsize(file_path) < 4096:   # 4KB
#                 corrupted_count += 1

# print("Corrupted (<4KB) files in train:", corrupted_count)


In [75]:
# peak_db_values = []

# for genre in tr:
#     for file_path in tr[genre]['vocals']:
#         y, _ = librosa.load(file_path, sr=SR)
#         peak = np.max(np.abs(y))

#         if peak > 0:
#             peak_db = 20 * np.log10(peak)
#             peak_db_values.append(peak_db)

# average_peak_db = np.mean(peak_db_values)

# print("Average Peak Amplitude (dB):", round(average_peak_db, 2))


In [76]:
# import numpy as np
# import librosa

# centroid_values = []

# for stem_name, file_list in tr['blues'].items():
#     for file_path in file_list:
#         y, _ = librosa.load(file_path, sr=SR)

#         centroid = librosa.feature.spectral_centroid(
#             y=y,
#             sr=SR,
#             n_fft=N_FFT,
#             hop_length=HOP_LENGTH
#         )

#         centroid_values.extend(centroid.flatten())

# mean_centroid = np.mean(centroid_values)

# print("Mean Spectral Centroid (Hz):", round(mean_centroid, 2))


In [77]:
# import numpy as np
# import librosa

# genre_centroids = {}

# for genre in tr:
#     centroid_values = []

#     for stem_name, file_list in tr[genre].items():
#         for file_path in file_list:
#             y, _ = librosa.load(file_path, sr=SR)

#             centroid = librosa.feature.spectral_centroid(
#                 y=y,
#                 sr=SR,
#                 n_fft=N_FFT,
#                 hop_length=HOP_LENGTH
#             )

#             centroid_values.extend(centroid.flatten())

#     genre_centroids[genre] = np.mean(centroid_values)

# # Print all means
# for g, val in genre_centroids.items():
#     print(g, round(val, 2))

# # Find genre with highest mean centroid
# highest_genre = max(genre_centroids, key=genre_centroids.get)

# print("\nGenre with highest mean spectral centroid:", highest_genre)

In [78]:
# import librosa
# import numpy as np

# count = 0
# threshold_samples = int(0.5 * SR)

# for genre in tr:
#     for stem_name in tr[genre]:
#         for file_path in tr[genre][stem_name]:

#             y, _ = librosa.load(file_path, sr=SR)

#             intervals = librosa.effects.split(y, top_db=TOP_DB)

#             if len(intervals) == 0:
#                 # fully silent file
#                 count += 1
#             else:
#                 first_start = intervals[0][0]

#                 if first_start >= threshold_samples:
#                     count += 1

# print("Files with silence in first 0.5 sec:", count)


In [79]:
# import os
# import numpy as np
# import pandas as pd
# import librosa
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.model_selection import train_test_split
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.metrics import f1_score, confusion_matrix, classification_report

# # --- 1. Setup and Preprocessing ---
# ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
# STEMS_PATH = os.path.join(ROOT, 'genres_stems')
# GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

# def extract_features(song_path):
#     # Load 10s at 22050Hz
#     y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
#     tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
#     spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
#     zcr = np.mean(librosa.feature.zero_crossing_rate(y))
#     rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
#     return [float(tempo), spec_cent, zcr, rolloff]

# # --- 2. Data Preparation & Stratified Split ---
# data = []
# for g in GENRES:
#     gp = os.path.join(STEMS_PATH, g)
#     songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
#     for s in songs[:50]: # Sampling 50 for speed; use all for final
#         data.append({'path': os.path.join(gp, s), 'genre': g})

# df = pd.DataFrame(data)
# train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)

# # --- 3. Model Training (Decision Tree) ---
# X_train = np.array([extract_features(p) for p in train_df['path']])
# y_train = train_df['genre']
# X_val = np.array([extract_features(p) for p in val_df['path']])
# y_val = val_df['genre']

# clf = DecisionTreeClassifier(max_depth=5, random_state=42)
# clf.fit(X_train, y_train)

In [80]:
# # --- Predictions ---
# y_pred = clf.predict(X_val)

# # --- Metrics ---
# macro_f1 = f1_score(y_val, y_pred, average='macro')
# accuracy = np.mean(y_pred == y_val)

# cm = confusion_matrix(y_val, y_pred, labels=GENRES)
# cr = classification_report(y_val, y_pred, output_dict=True)

# print(f"Validation Macro F1 Score: {macro_f1:.4f}")
# print(f"Model Accuracy: {accuracy:.4f}\n")

# print("Detailed Classification Report:")
# print(classification_report(y_val, y_pred, digits=4))

# # --- Precision of hiphop ---
# hiphop_precision = cr['hiphop']['precision']
# print("\nPrecision of hiphop:", round(hiphop_precision, 4))

# # --- Recall of pop ---
# pop_recall = cr['pop']['recall']
# print("Recall of pop:", round(pop_recall, 4))


In [81]:
# plt.figure(figsize=(10,8))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=GENRES, yticklabels=GENRES)
# plt.xlabel("Predicted")
# plt.ylabel("Actual")
# plt.title("Confusion Matrix")
# plt.show()


In [82]:
# print("\nTP, TN, FP, FN per genre:\n")

# tp_dict = {}
# fn_dict = {}

# for i, genre in enumerate(GENRES):
#     TP = cm[i, i]
#     FN = np.sum(cm[i, :]) - TP
#     FP = np.sum(cm[:, i]) - TP
#     TN = np.sum(cm) - (TP + FN + FP)

#     tp_dict[genre] = TP
#     fn_dict[genre] = FN

#     print(f"{genre}: TP={TP}, TN={TN}, FP={FP}, FN={FN}")

# # --- Highest TP ---
# highest_tp_genre = max(tp_dict, key=tp_dict.get)
# print("\nGenre with highest True Positives:", highest_tp_genre)

# # --- Lowest FN ---
# lowest_fn_genre = min(fn_dict, key=fn_dict.get)
# print("Genre with lowest False Negatives:", lowest_fn_genre)

# Milestone 3

1. You have a batch of grayscale spectrogram images represented as a PyTorch tensor with shape (32, 1, 64, 64). You need to flatten this tensor (excluding the batch dimension) to feed it into a Fully Connected (Linear) layer. What is the size of the input features for that Linear layer?

In [83]:
# import torch

# # Example tensor
# x = torch.randn(32, 1, 64, 64)

# # Flatten except batch dimension
# x_flat = x.view(x.size(0), -1)

# print("Original shape:", x.shape)
# print("Flattened shape:", x_flat.shape)
# print("Input features for Linear layer:", x_flat.shape[1])


2. You are converting a 1-second audio clip (Sample Rate = 16,000 Hz) into a Mel-Spectrogram using torchaudio. The configuration is n_fft=400, hop_length=160, n_mels=64. Based on your code output, what is the resulting tensor shape?

In [84]:
# import torch
# import torchaudio

# # 1-second audio at 16kHz
# sample_rate = 16000
# waveform = torch.randn(1, sample_rate)  # (channels, samples)

# # MelSpectrogram configuration
# mel_transform = torchaudio.transforms.MelSpectrogram(
#     sample_rate=sample_rate,
#     n_fft=400,
#     hop_length=160,
#     n_mels=64
# )

# # Generate Mel Spectrogram
# mel_spec = mel_transform(waveform)

# print("Waveform shape:", waveform.shape)
# print("Mel Spectrogram shape:", mel_spec.shape)


3. Your CNN passes a (1, 64, 64) input through nn.Conv2d(..., kernel_size=3, stride=1, padding=1) followed by nn.MaxPool2d(kernel_size=2, stride=2). What is the spatial output shape (Height, Width)?

In [85]:
# import torch
# import torch.nn as nn

# # Input tensor: (batch_size, channels, height, width)
# x = torch.randn(1, 1, 64, 64)

# # Define layers
# conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
# pool = nn.MaxPool2d(kernel_size=2, stride=2)

# # Forward pass
# x = conv(x)
# print("After Conv2d:", x.shape)

# x = pool(x)
# print("After MaxPool2d:", x.shape)


4. You have a hidden layer defined as nn.Linear(in_features=128, out_features=64). How many total learnable parameters (weights + biases) does this specific layer contain?

In [86]:
# import torch
# import torch.nn as nn

# # Define the Linear layer
# layer = nn.Linear(in_features=128, out_features=64)

# # Total learnable parameters
# total_params = sum(p.numel() for p in layer.parameters())

# print("Weight shape:", layer.weight.shape)
# print("Bias shape:", layer.bias.shape)
# print("Total learnable parameters:", total_params)


5. Your training dataset consists of 1,050 audio files. You initialize a DataLoader with batch_size=32 and drop_last=False. How many batches will run in one epoch?

In [87]:
# from torch.utils.data import DataLoader, TensorDataset
# import torch

# # Dummy dataset
# dataset = TensorDataset(torch.randn(1050, 10))

# loader = DataLoader(dataset, batch_size=32, drop_last=False)

# print("Number of batches per epoch:", len(loader))


6. You are performing a 3-class classification. The model outputs logits [2.5, 1.0, 0.1] for a target of Class 0. Using nn.CrossEntropyLoss, what is the calculated loss value (rounded to 3 decimals)?

In [88]:
# import torch
# import torch.nn as nn

# # Logits (no softmax applied)
# logits = torch.tensor([[2.5, 1.0, 0.1]])  # shape (1, 3)

# # Target class index
# target = torch.tensor([0])  # Class 0

# # Loss function
# criterion = nn.CrossEntropyLoss()

# loss = criterion(logits, target)

# print("Loss value:", round(loss.item(), 3))


7. You are training with SGD. Current Weight = 0.5, Gradient = 0.2, Learning Rate = 0.01. What is the new weight value after one optimizer step?

In [89]:
# import torch

# # Given values
# weight = torch.tensor(0.5, requires_grad=True)
# gradient = torch.tensor(0.2)
# learning_rate = 0.01

# # Manually assign gradient
# weight.grad = gradient

# # SGD update rule: w = w - lr * grad
# with torch.no_grad():
#     weight -= learning_rate * weight.grad

# print("Updated weight:", round(weight.item(), 3))


8. You want to apply a convolution with kernel_size=5 and stride=1 such that the output size remains the same as the input. What padding value must you set?

In [90]:
# # Given values
# kernel_size = 5
# stride = 1

# # Compute padding for same output size
# if stride == 1:
#     padding = (kernel_size - 1) // 2
# else:
#     raise ValueError("This simple formula works for stride=1 only.")

# print("Required padding:", padding)


10. After training, you evaluate your model on a test set of 200 spectrograms and it correctly predicts 170 of them. What is the accuracy?

In [91]:
# correct = 170
# total = 200

# accuracy = correct / total
# accuracy_percent = accuracy * 100

# print("Accuracy:", accuracy)
# print("Accuracy (%):", accuracy_percent)
